# Command job with MLFlow

In [50]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import dotenv
import os

dotenv.load_dotenv(override=True)

ml_client = ml_client = MLClient(
     credential         = DefaultAzureCredential()
    ,subscription_id    = os.environ["AZURE_SUBSCRIPTION_ID"]
    ,resource_group_name= os.environ["AZURE_RESOURCE_GROUP"]
    ,workspace_name     = os.environ["AZURE_ML"]
)


Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


# Create an appropriate environment

In [51]:
for env in ml_client.environments.list():
    print(env.name)

diabetic_prediction_environment
docker-image-plus-conda-example
aml-docker-image-example
AzureML-ACPT-pytorch-1.13-py38-cuda11.7-gpu


In [52]:
from azure.ai.ml.entities import Environment

environment_name = "diabetic_prediction_environment"
environment_version = "4"

# Get the environment if available
try: 
    environment = ml_client.environments.get(name=environment_name, label="latest")
except: 
    environment = None

# Get the latest version of the environment
try:
    version = environment.version
except:
    version = None


if environment is not None and version != environment_version:

    environment = Environment(
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04"
        ,conda_file = "./env/conda.yml"
        ,name = environment_name
        ,description = "Environment for diabetic classification - Associate Data Science Azure"
        ,version = environment_version
    )

    print(f"Creating environment ... {environment_name}:{environment_version}")
    ml_client.create_or_update(environment)

else:
    environment = ml_client.environments.get(name=environment_name, label="latest")

Creating environment ... diabetic_prediction_environment:4


# Log with MLFlow

Depending on the type of value you want to log, use the MLflow command to store the metric with the experiment run:

- ```mlflow.log_param()```: Log single key-value parameter. Use this function for an input parameter you want to log.
- ```mlflow.log_metric()```: Log single key-value metric. Value must be a number. Use this function for any output you want to store with the run.
- ```mlflow.log_artifact()```: Log a file. Use this function for any plot you want to log, save as image file first.

In [53]:
from azure.ai.ml import command
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

data_asset_name     = "diabetes-data-ml-table"
data_asset_version  = "0"

job = command(
     code               = "./code"
    ,command            = "python train.py --data-asset ${{inputs.DATA}}"
    ,inputs             = {"DATA": Input(type=AssetTypes.MLTABLE, path=f"azureml:{data_asset_name}:{data_asset_version}")}
    ,environment        = environment_name + "@latest"
    ,compute            = "diabetes-cluster"
    ,display_name       = "Run the diabetes job with a custom environment"
    ,experiment_name    = "train-diabetes-model-job"
)

poller = ml_client.create_or_update(job)

# Retrieve metrics

In [54]:
import mlflow

mlflow.set_tracking_uri(ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri)

In [55]:
for exp in mlflow.search_experiments():
    print(exp.name)

experiment = mlflow.get_experiment_by_name("train-diabetes-model-job")

diabetes-training
prepare_image
auto-ml-class-dev
me-very-own-fecking-experiment
train-diabetes-model-job


In [ ]:
mlflow.search_runs(exp.experiment_id).sort_values("start_time", ascending=False).head()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.accuracy,params.n_jobs,params.C,params.max_iter,...,params.tol,params.intercept_scaling,params.dual,params.class_weight,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.rootRunId,tags.estimator_class,tags.mlflow.autologging,tags.estimator_name
14,hungry_receipt_lm4wqbpnb7,ea4358f8-61ae-487f-bf41-dc29e599c658,FAILED,,2026-03-12 14:45:14.239000+00:00,2026-03-12 14:45:36.138000+00:00,NaN,None,None,None,...,None,None,None,None,Run the diabetes job with a custom environment,Dries Gouwy,hungry_receipt_lm4wqbpnb7,None,None,None
12,musing_snake_pw2h0815x7,ea4358f8-61ae-487f-bf41-dc29e599c658,FAILED,,2026-03-12 14:44:41.052000+00:00,2026-03-12 14:44:59.239000+00:00,NaN,None,None,None,...,None,None,None,None,Run the diabetes job with a custom environment,Dries Gouwy,musing_snake_pw2h0815x7,None,None,None
13,neat_bottle_6n4b2mgt4h,ea4358f8-61ae-487f-bf41-dc29e599c658,FAILED,,2026-03-12 14:41:32.256000+00:00,2026-03-12 14:44:23.015000+00:00,NaN,None,None,None,...,None,None,None,None,Run the diabetes job with a custom environment,Dries Gouwy,neat_bottle_6n4b2mgt4h,None,None,None
11,c0e40c4e-4d23-42e7-9e53-f6a58edc4090,ea4358f8-61ae-487f-bf41-dc29e599c658,FINISHED,,2026-03-12 14:20:22.975000+00:00,2026-03-12 14:20:26.909000+00:00,0.797667,None,None,None,...,None,None,None,None,nifty_carrot_l0jxj7qq,Dries Gouwy,c0e40c4e-4d23-42e7-9e53-f6a58edc4090,None,None,None
10,careful_leaf_z1bg719xrw,ea4358f8-61ae-487f-bf41-dc29e599c658,FINISHED,,2026-03-12 14:17:49.142000+00:00,2026-03-12 14:20:35.068000+00:00,NaN,None,1.0,100,...,0.0001,1,False,None,Run the diabetes job with a custom environment,Dries Gouwy,careful_leaf_z1bg719xrw,sklearn.linear_model._logistic.LogisticRegression,sklearn,LogisticRegression


: 